<a href="https://colab.research.google.com/github/Srihas03/Srihas_INFO5731_Spring2026/blob/main/Srihas_Sankidi_Assignment_1_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [ ]:

!pip -q install requests pandas tqdm

import time, random
import requests
import pandas as pd
from tqdm import tqdm
from google.colab import files

QUERY = "machine learning"
TARGET = 10000
OUT_CSV = "semantic_scholar_abstracts.csv"

# OPTIONAL API KEY (recommended but not required)
S2_API_KEY = ""  # paste if you have one

LIMIT = 100                    # max is typically 100
DELAY_BETWEEN_CALLS = 1.0      # increase to 2-3 if rate-limited

BASE_URL = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
FIELDS = "paperId,title,abstract,year,venue,authors,url,citationCount"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
})
if S2_API_KEY.strip():
    session.headers["x-api-key"] = S2_API_KEY.strip()

def get_with_debug(url, params=None, timeout=30):
    r = session.get(url, params=params, timeout=timeout)
    return r

def robust_get(url, params=None, max_retries=12, timeout=30):
    last_exc = None
    last_resp = None

    for attempt in range(max_retries):
        try:
            r = get_with_debug(url, params=params, timeout=timeout)
            last_resp = r

            if r.status_code == 200:
                return r

            # Retry on throttling/transient errors
            if r.status_code in (429, 500, 502, 503, 504):
                ra = r.headers.get("Retry-After")
                if ra:
                    try:
                        sleep_s = float(ra)
                    except:
                        sleep_s = (2 ** attempt) + random.uniform(0, 0.5)
                else:
                    sleep_s = (2 ** attempt) + random.uniform(0, 0.5)

                time.sleep(min(sleep_s, 90))
                continue

            # For other codes (400/403/etc), fail with response text
            raise Exception(f"HTTP {r.status_code}: {r.text[:500]}")

        except Exception as e:
            last_exc = e
            sleep_s = (2 ** attempt) + random.uniform(0, 0.5)
            time.sleep(min(sleep_s, 60))

    # If we exhausted retries, show the last response if available
    if last_resp is not None:
        raise Exception(f"Failed after retries. Last status={last_resp.status_code}. Body={last_resp.text[:800]}")
    raise Exception(f"Failed after retries. Last exception={last_exc}")

def collect_abstracts_bulk(query, target):
    token = None
    rows = []
    seen = set()

    pbar = tqdm(total=target, desc="Abstracts collected", unit="abs")

    while len(rows) < target:
        params = {
            "query": query,
            "fields": FIELDS,
            "limit": LIMIT
        }
        if token:
            params["token"] = token  # bulk endpoint supports token continuation

        r = robust_get(BASE_URL, params=params)
        data = r.json()

        papers = data.get("data", []) or []
        if not papers:
            break

        for p in papers:
            pid = p.get("paperId")
            if not pid or pid in seen:
                continue
            seen.add(pid)

            abs_txt = p.get("abstract")
            if not abs_txt or not str(abs_txt).strip():
                continue

            authors = p.get("authors", []) or []
            author_names = "; ".join([a.get("name","") for a in authors if a.get("name")])

            rows.append({
                "paperId": pid,
                "title": (p.get("title") or "").strip(),
                "abstract": abs_txt.strip(),
                "year": p.get("year"),
                "venue": p.get("venue"),
                "authors": author_names,
                "url": p.get("url"),
                "citationCount": p.get("citationCount"),
                "query": query
            })

            pbar.update(1)
            if len(rows) >= target:
                break

        # continuation token
        token = data.get("token")
        if not token:
            break

        time.sleep(DELAY_BETWEEN_CALLS)

    pbar.close()
    return pd.DataFrame(rows)

# RUN
df = collect_abstracts_bulk(QUERY, TARGET)
print(f"\nCollected {len(df)} abstracts.")
df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print("Saved:", OUT_CSV)

files.download(OUT_CSV)
df.head()



Abstracts collected: 100%|██████████| 10000/10000 [00:38<00:00, 257.24abs/s]



Collected 10000 abstracts.
Saved: semantic_scholar_abstracts.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,paperId,title,abstract,year,venue,authors,url,citationCount,query
0,00000c33779acab142af6c7a6dae8b36fac0805d,Insights into Household Electric Vehicle Charg...,In the era of burgeoning electric vehicle (EV)...,2024.0,Energies,Ahmad Almaghrebi; Kevin James; Fares al Juhesh...,https://www.semanticscholar.org/paper/00000c33...,15,machine learning
1,0000238f07f151172cf2602588ba762b55c8464b,Personalized Prediction of Response to Smartph...,Background Meditation apps have surged in popu...,2021.0,Journal of Medical Internet Research,Christian A. Webb; M. Hirshberg; R. Davidson; ...,https://www.semanticscholar.org/paper/0000238f...,16,machine learning
2,0000315635be19f6278dbc72597b3065fac405f0,Abstractive text summarization of low-resource...,Background Humans must be able to cope with th...,2023.0,PeerJ Computer Science,Nida Shafiq; Isma Hamid; Muhammad Asif; Qamar ...,https://www.semanticscholar.org/paper/00003156...,22,machine learning
3,00005d68c6c7eb4d3c27da8242a30b9a498f991e,Detection of DDoS Attacks on Clouds Computing ...,The growing number of cloud-based services has...,2023.0,International Conference on Communication and ...,Iehab Alrassan; Asma Alqahtani,https://www.semanticscholar.org/paper/00005d68...,6,machine learning
4,00005f1b7e976068ca4b5a1b546d9945158b3bfc,Diffusion Generative Models for Designing Effi...,"Diffusion generative models, a class of machin...",2024.0,Journal of Physical Chemistry A,Lasse Kreimendahl; Mikhail Karnaukh; M. Röhr,https://www.semanticscholar.org/paper/00005f1b...,1,machine learning


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [ ]:
# ================================
# TEXT CLEANING PIPELINE (Colab)
# Prints output for each step + saves CSV with new clean column
# ================================
!pip -q install pandas scikit-learn nltk

import os, re
import pandas as pd
from google.colab import files
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

import nltk
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

# --------- 0) LOAD CSV (upload if missing) ----------
INPUT_CSV = "YOUR_FILE.csv"   # <-- CHANGE THIS to your file name if you know it
TEXT_COLUMN = None            # <-- set to column name if you know it (e.g., "abstract" or "review")

if not os.path.exists(INPUT_CSV):
    print(f"File '{INPUT_CSV}' not found. Please upload your collected CSV now...")
    uploaded = files.upload()
    # pick the first uploaded file as input
    INPUT_CSV = next(iter(uploaded.keys()))
    print("Using uploaded file:", INPUT_CSV)

df = pd.read_csv(INPUT_CSV)
print("\nLoaded:", INPUT_CSV)
print("Shape:", df.shape)
print("Columns:", list(df.columns))

# Auto-detect a text column if TEXT_COLUMN not provided
if TEXT_COLUMN is None:
    # Common candidates (add more if needed)
    candidates = ["review", "reviews", "text", "content", "abstract", "summary", "body", "comment"]
    found = None
    for c in candidates:
        if c in df.columns:
            found = c
            break
    if found is None:
        # fallback: choose the first object (string) column
        obj_cols = [c for c in df.columns if df[c].dtype == "object"]
        if not obj_cols:
            raise ValueError("No text-like column found. Set TEXT_COLUMN to your text column name.")
        found = obj_cols[0]
    TEXT_COLUMN = found

print("Using TEXT_COLUMN =", TEXT_COLUMN)

# Raw text column
df["raw_text"] = df[TEXT_COLUMN].fillna("").astype(str)

print("\n--- RAW sample (2 rows) ---")
for i, t in enumerate(df["raw_text"].head(2).tolist(), 1):
    print(f"\n[{i}] {t[:300]}")

# --------- Helpers ----------
def normalize_spaces(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

# (1) Remove noise: special characters and punctuations
def step1_remove_noise(s: str) -> str:
    # keep letters/numbers/underscore/whitespace; remove punctuation/special chars
    s = re.sub(r"[^\w\s]", " ", s)
    return normalize_spaces(s)

# (2) Remove numbers
def step2_remove_numbers(s: str) -> str:
    s = re.sub(r"\d+", " ", s)
    return normalize_spaces(s)

# (4) Lowercase
def step4_lowercase(s: str) -> str:
    return s.lower()

# (3) Remove stopwords (offline list from sklearn)
STOPWORDS = set(ENGLISH_STOP_WORDS)
def step3_remove_stopwords(s: str) -> str:
    tokens = s.split()
    tokens = [w for w in tokens if w not in STOPWORDS]
    return " ".join(tokens)

# (5) Stemming (offline)
stemmer = PorterStemmer()
def step5_stemming(s: str) -> str:
    tokens = s.split()
    tokens = [stemmer.stem(w) for w in tokens]
    return " ".join(tokens)

# (6) Lemmatization
# Try WordNetLemmatizer (needs wordnet data). If missing, fallback.
lemmatizer = WordNetLemmatizer()

def simple_rule_lemmatize_word(w: str) -> str:
    # Lightweight fallback: common English suffix rules (not as accurate as WordNet)
    if len(w) > 4 and w.endswith("ies"):   return w[:-3] + "y"
    if len(w) > 4 and w.endswith("sses"):  return w[:-2]
    if len(w) > 3 and w.endswith("s") and not w.endswith("ss"):  return w[:-1]
    if len(w) > 5 and w.endswith("ing"):   return w[:-3]
    if len(w) > 4 and w.endswith("ed"):    return w[:-2]
    return w

def step6_lemmatization(s: str) -> str:
    tokens = s.split()
    out = []
    for w in tokens:
        try:
            out.append(lemmatizer.lemmatize(w))
        except LookupError:
            # WordNet data missing -> fallback
            out.append(simple_rule_lemmatize_word(w))
    return " ".join(out)

# --------- APPLY + PRINT OUTPUT FOR EACH PART ----------
# Step order per your list:
# (1) noise, (2) numbers, (3) stopwords, (4) lowercase, (5) stemming, (6) lemmatization
# Practical order: noise -> numbers -> lowercase -> stopwords -> stemming/lemmatization

df["s1_no_noise"] = df["raw_text"].apply(step1_remove_noise)
print("\n\n(1) Remove noise (special characters & punctuations) - OUTPUT (2 rows)")
for i in range(min(2, len(df))):
    print(f"\n[{i+1}] BEFORE: {df['raw_text'].iloc[i][:200]}")
    print(f"[{i+1}] AFTER : {df['s1_no_noise'].iloc[i][:200]}")

df["s2_no_numbers"] = df["s1_no_noise"].apply(step2_remove_numbers)
print("\n\n(2) Remove numbers - OUTPUT (2 rows)")
for i in range(min(2, len(df))):
    print(f"\n[{i+1}] BEFORE: {df['s1_no_noise'].iloc[i][:200]}")
    print(f"[{i+1}] AFTER : {df['s2_no_numbers'].iloc[i][:200]}")

df["s4_lower"] = df["s2_no_numbers"].apply(step4_lowercase)
print("\n\n(4) Lowercase all texts - OUTPUT (2 rows)")
for i in range(min(2, len(df))):
    print(f"\n[{i+1}] BEFORE: {df['s2_no_numbers'].iloc[i][:200]}")
    print(f"[{i+1}] AFTER : {df['s4_lower'].iloc[i][:200]}")

df["s3_no_stopwords"] = df["s4_lower"].apply(step3_remove_stopwords)
print("\n\n(3) Remove stopwords - OUTPUT (2 rows)")
for i in range(min(2, len(df))):
    print(f"\n[{i+1}] BEFORE: {df['s4_lower'].iloc[i][:200]}")
    print(f"[{i+1}] AFTER : {df['s3_no_stopwords'].iloc[i][:200]}")

df["s5_stem"] = df["s3_no_stopwords"].apply(step5_stemming)
print("\n\n(5) Stemming - OUTPUT (2 rows)")
for i in range(min(2, len(df))):
    print(f"\n[{i+1}] BEFORE: {df['s3_no_stopwords'].iloc[i][:200]}")
    print(f"[{i+1}] AFTER : {df['s5_stem'].iloc[i][:200]}")

df["s6_lemma"] = df["s3_no_stopwords"].apply(step6_lemmatization)
print("\n\n(6) Lemmatization - OUTPUT (2 rows)")
for i in range(min(2, len(df))):
    print(f"\n[{i+1}] BEFORE: {df['s3_no_stopwords'].iloc[i][:200]}")
    print(f"[{i+1}] AFTER : {df['s6_lemma'].iloc[i][:200]}")

# --------- FINAL CLEAN COLUMN (choose lemmatized as final) ----------
df["clean_text"] = df["s6_lemma"]

OUT_CSV = os.path.splitext(INPUT_CSV)[0] + "_cleaned.csv"
df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print("\nSaved cleaned file:", OUT_CSV)

files.download(OUT_CSV)
df.head(3)

File 'YOUR_FILE.csv' not found. Please upload your collected CSV now...


Saving semantic_scholar_abstracts.csv to semantic_scholar_abstracts (1).csv
Using uploaded file: semantic_scholar_abstracts (1).csv

Loaded: semantic_scholar_abstracts (1).csv
Shape: (10000, 9)
Columns: ['paperId', 'title', 'abstract', 'year', 'venue', 'authors', 'url', 'citationCount', 'query']
Using TEXT_COLUMN = abstract

--- RAW sample (2 rows) ---

[1] In the era of burgeoning electric vehicle (EV) popularity, understanding the patterns of EV users’ behavior is imperative. This paper examines the trends in household charging sessions’ timing, duration, and energy consumption by analyzing real-world residential charging data. By leveraging the info

[2] Background Meditation apps have surged in popularity in recent years, with an increasing number of individuals turning to these apps to cope with stress, including during the COVID-19 pandemic. Meditation apps are the most commonly used mental health apps for depression and anxiety. However, little 


(1) Remove noise (special chara

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,paperId,title,abstract,year,venue,authors,url,citationCount,query,raw_text,s1_no_noise,s2_no_numbers,s4_lower,s3_no_stopwords,s5_stem,s6_lemma,clean_text
0,00000c33779acab142af6c7a6dae8b36fac0805d,Insights into Household Electric Vehicle Charg...,In the era of burgeoning electric vehicle (EV)...,2024.0,Energies,Ahmad Almaghrebi; Kevin James; Fares al Juhesh...,https://www.semanticscholar.org/paper/00000c33...,15,machine learning,In the era of burgeoning electric vehicle (EV)...,In the era of burgeoning electric vehicle EV p...,In the era of burgeoning electric vehicle EV p...,in the era of burgeoning electric vehicle ev p...,era burgeoning electric vehicle ev popularity ...,era burgeon electr vehicl ev popular understan...,era burgeoning electric vehicle ev popularity ...,era burgeoning electric vehicle ev popularity ...
1,0000238f07f151172cf2602588ba762b55c8464b,Personalized Prediction of Response to Smartph...,Background Meditation apps have surged in popu...,2021.0,Journal of Medical Internet Research,Christian A. Webb; M. Hirshberg; R. Davidson; ...,https://www.semanticscholar.org/paper/0000238f...,16,machine learning,Background Meditation apps have surged in popu...,Background Meditation apps have surged in popu...,Background Meditation apps have surged in popu...,background meditation apps have surged in popu...,background meditation apps surged popularity r...,background medit app surg popular recent year ...,background meditation apps surged popularity r...,background meditation apps surged popularity r...
2,0000315635be19f6278dbc72597b3065fac405f0,Abstractive text summarization of low-resource...,Background Humans must be able to cope with th...,2023.0,PeerJ Computer Science,Nida Shafiq; Isma Hamid; Muhammad Asif; Qamar ...,https://www.semanticscholar.org/paper/00003156...,22,machine learning,Background Humans must be able to cope with th...,Background Humans must be able to cope with th...,Background Humans must be able to cope with th...,background humans must be able to cope with th...,background humans able cope huge amounts infor...,background human abl cope huge amount inform p...,background human able cope huge amount informa...,background human able cope huge amount informa...


# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [ ]:
# ============================================
# FULL GOOGLE COLAB IMPLEMENTATION
# Syntax & Structure Analysis: POS + Parsing + NER
# Using STANZA
# ============================================

!pip -q install pandas stanza tqdm

import os
import pandas as pd
import stanza
from collections import Counter
from tqdm import tqdm
from google.colab import files

# ------------------------------------------------
# 0) Upload cleaned CSV
# ------------------------------------------------
print("Upload your cleaned CSV file (semantic_scholar_abstracts_cleaned.csv)")
uploaded = files.upload()
CSV_FILE = next(iter(uploaded.keys()))
print("Using:", CSV_FILE)

# ------------------------------------------------
# 1) Load dataset
# ------------------------------------------------
df = pd.read_csv(CSV_FILE)
print("\nLoaded:", CSV_FILE)
print("Shape:", df.shape)
print("Columns:", list(df.columns))

# Choose text column:
# - Use "raw_text" or "abstract" for BEST syntax/NER
# - Use "clean_text" if your assignment requires it
TEXT_COL = "clean_text"
if TEXT_COL not in df.columns:
    # fallback to abstract/raw_text if clean_text isn't present
    if "raw_text" in df.columns:
        TEXT_COL = "raw_text"
    elif "abstract" in df.columns:
        TEXT_COL = "abstract"
    else:
        raise ValueError("No suitable text column found (clean_text/raw_text/abstract).")

print("Using TEXT COLUMN:", TEXT_COL)
texts = df[TEXT_COL].fillna("").astype(str).tolist()

# ------------------------------------------------
# 2) Download Stanza models (needs internet in Colab)
# ------------------------------------------------
stanza.download("en")  # one-time download

nlp = stanza.Pipeline(
    "en",
    processors="tokenize,pos,lemma,depparse,constituency,ner",
    tokenize_no_ssplit=False,
    verbose=False
)

def process_text(t: str):
    t = (t or "").strip()
    if not t:
        return None
    return nlp(t)

# ------------------------------------------------
# SETTINGS (Printing all trees in notebook will crash it)
# We save ALL trees to files, and print only 1-3 sentences on screen.
# ------------------------------------------------
PRINT_SENTENCES_ON_SCREEN = 3

# Output files (downloadable)
POS_OUT = "pos_counts.txt"
CONSTIT_OUT = "constituency_trees_all.txt"
DEP_OUT = "dependency_trees_all.txt"
NER_SUMMARY_OUT = "ner_category_counts.csv"
NER_TOP_OUT = "ner_top_entities.csv"

# ============================================
# (1) POS TAGGING
# ============================================
pos_counts = Counter()
upos_full = Counter()

print("\n==============================")
print("(1) POS TAGGING + COUNTS")
print("==============================")

for t in tqdm(texts, desc="POS tagging", unit="doc"):
    doc = process_text(t)
    if doc is None:
        continue

    for sent in doc.sentences:
        for w in sent.words:
            upos_full[w.upos] += 1
            if w.upos in ("NOUN", "PROPN"):
                pos_counts["Noun"] += 1
            elif w.upos == "VERB":
                pos_counts["Verb"] += 1
            elif w.upos == "ADJ":
                pos_counts["Adjective"] += 1
            elif w.upos == "ADV":
                pos_counts["Adverb"] += 1

# Save POS results
with open(POS_OUT, "w", encoding="utf-8") as f:
    f.write("=== POS COUNTS (Requested) ===\n")
    f.write(f"Noun      : {pos_counts['Noun']}\n")
    f.write(f"Verb      : {pos_counts['Verb']}\n")
    f.write(f"Adjective : {pos_counts['Adjective']}\n")
    f.write(f"Adverb    : {pos_counts['Adverb']}\n\n")

    f.write("=== Top 15 UPOS tags (Extra) ===\n")
    for tag, c in upos_full.most_common(15):
        f.write(f"{tag:>6} : {c}\n")

print("POS totals:")
print("  Noun     :", pos_counts["Noun"])
print("  Verb     :", pos_counts["Verb"])
print("  Adj      :", pos_counts["Adjective"])
print("  Adv      :", pos_counts["Adverb"])
print("Saved POS counts file:", POS_OUT)

# ============================================
# (2) CONSTITUENCY + DEPENDENCY PARSING
# Save ALL trees to files; print a few on screen
# ============================================
print("\n==============================================")
print("(2) CONSTITUENCY + DEPENDENCY PARSING")
print("==============================================")

def dependency_lines(sent):
    lines = []
    for w in sent.words:
        head = "ROOT" if w.head == 0 else sent.words[w.head - 1].text
        lines.append(f"{w.text:>15} -> {head:>15}  [{w.deprel}]")
    return lines

example_sentence = None
example_constituency = None
example_dependency = None

screen_printed = 0
sent_counter = 0

with open(CONSTIT_OUT, "w", encoding="utf-8") as fc, open(DEP_OUT, "w", encoding="utf-8") as fd:
    for doc_i, t in enumerate(tqdm(texts, desc="Parsing", unit="doc"), start=1):
        doc = process_text(t)
        if doc is None:
            continue

        for sent in doc.sentences:
            sent_counter += 1

            # Constituency tree
            fc.write(f"\n=== DOC {doc_i} | SENT {sent_counter} ===\n")
            fc.write(sent.text + "\n")
            fc.write(str(sent.constituency) + "\n")

            # Dependency tree
            fd.write(f"\n=== DOC {doc_i} | SENT {sent_counter} ===\n")
            fd.write(sent.text + "\n")
            fd.write("Dependency arcs (word -> head [relation]):\n")
            dep = dependency_lines(sent)
            fd.write("\n".join(dep) + "\n")

            # Save one example for explanation
            if example_sentence is None and sent.text.strip():
                example_sentence = sent.text
                example_constituency = str(sent.constituency)
                example_dependency = dep

            # Print a few trees on screen only
            if screen_printed < PRINT_SENTENCES_ON_SCREEN:
                print("\n---------------------------------")
                print(f"Sentence Example #{screen_printed+1}: {sent.text}")

                print("\nConstituency tree:")
                print(str(sent.constituency))

                print("\nDependency arcs (first 25):")
                for line in dep[:25]:
                    print(line)

                screen_printed += 1

print("\nSaved ALL constituency trees to:", CONSTIT_OUT)
print("Saved ALL dependency trees to:", DEP_OUT)

# Explanation for one sentence
print("\n--- EXPLANATION (One sentence example) ---")
print("Sentence:", example_sentence)

print("\nConstituency parsing tree (phrase structure):")
print(example_constituency)
print("\nExplanation:")
print(
    "Constituency parsing groups words into nested phrases such as NP (noun phrase), "
    "VP (verb phrase), and PP (prepositional phrase). It shows the phrase structure "
    "of the entire sentence (how words combine into phrases)."
)

print("\nDependency parsing (relations between words):")
print("\n".join(example_dependency[:25]))
print("\nExplanation:")
print(
    "Dependency parsing links each word to a HEAD word using labeled relations. "
    "Labels like nsubj (subject), obj (object), amod (adjective modifier) describe "
    "the grammatical role of each word. These arcs form a dependency tree."
)

# ============================================
# (3) NAMED ENTITY RECOGNITION (NER)
# ============================================
print("\n==============================")
print("(3) NAMED ENTITY RECOGNITION")
print("==============================")

label_map = {
    "PERSON": "Person",
    "ORG": "Organization",
    "GPE": "Location",
    "LOC": "Location",
    "PRODUCT": "Product",
    "DATE": "Date"
}

entity_type_counts = Counter()
entity_text_counts = Counter()

for t in tqdm(texts, desc="NER", unit="doc"):
    doc = process_text(t)
    if doc is None:
        continue

    for ent in doc.ents:
        if ent.type in label_map:
            cat = label_map[ent.type]
            entity_type_counts[cat] += 1
            entity_text_counts[(cat, ent.text)] += 1

print("\nEntity counts by category:")
for k in ["Person", "Organization", "Location", "Product", "Date"]:
    print(f"  {k:>12} : {entity_type_counts[k]}")

# Save NER results
ner_category_df = pd.DataFrame(
    [{"entity_category": k, "count": v} for k, v in entity_type_counts.items()]
).sort_values("count", ascending=False)

ner_top_df = pd.DataFrame(
    [{"entity_category": cat, "entity_text": txt, "count": c}
     for (cat, txt), c in entity_text_counts.most_common(2000)]
)

ner_category_df.to_csv(NER_SUMMARY_OUT, index=False, encoding="utf-8")
ner_top_df.to_csv(NER_TOP_OUT, index=False, encoding="utf-8")

print("\nSaved NER files:", NER_SUMMARY_OUT, "and", NER_TOP_OUT)

# ------------------------------------------------
# 4) Download outputs
# ------------------------------------------------
files.download(POS_OUT)
files.download(CONSTIT_OUT)
files.download(DEP_OUT)
files.download(NER_SUMMARY_OUT)
files.download(NER_TOP_OUT)

print("\nDONE ✅ All outputs saved + downloaded.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.2/337.2 kB 9.6 MB/s eta 0:00:00
Upload your cleaned CSV file (semantic_scholar_abstracts_cleaned.csv)


Saving semantic_scholar_abstracts (1)_cleaned.csv to semantic_scholar_abstracts (1)_cleaned.csv
Using: semantic_scholar_abstracts (1)_cleaned.csv

Loaded: semantic_scholar_abstracts (1)_cleaned.csv
Shape: (10000, 17)
Columns: ['paperId', 'title', 'abstract', 'year', 'venue', 'authors', 'url', 'citationCount', 'query', 'raw_text', 's1_no_noise', 's2_no_numbers', 's4_lower', 's3_no_stopwords', 's5_stem', 's6_lemma', 'clean_text']
Using TEXT COLUMN: clean_text


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources



(1) POS TAGGING + COUNTS


POS tagging:   4%|▎         | 366/10000 [1:10:57<31:07:38, 11.63s/doc]


KeyboardInterrupt: 

# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

In [6]:
# =========================================
# Google Colab Full Implementation (Fixed)
# GitHub Marketplace Actions Scraper + DQ + Preprocessing + TF-IDF
# =========================================

# 1) Install dependencies
!pip -q install beautifulsoup4 lxml nltk scikit-learn pandas requests

import time
import random
import re
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

# 2) NLTK downloads (FIXED: includes punkt_tab)
import nltk
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")   # ✅ FIX for your error
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

print("✅ Setup complete (libraries + NLTK data).")


# -----------------------------------------
# 3) Configuration
# -----------------------------------------
BASE = "https://github.com"
START_URL = "https://github.com/marketplace?type=actions"

TARGET_ROWS = 1000
START_PAGE = 1
MAX_PAGES = 800

# Increase delay if you see 403/429
DELAY_RANGE = (1.0, 2.0)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

STOP = set(stopwords.words("english"))
LEM = WordNetLemmatizer()


# -----------------------------------------
# 4) Fetch with retries
# -----------------------------------------
def fetch_with_retries(url, session, retries=4, timeout=30):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, headers=HEADERS, timeout=timeout)
            if r.status_code >= 400:
                raise requests.HTTPError(f"HTTP {r.status_code} for {url}")
            return r.text
        except Exception as e:
            last_err = e
            backoff = (2 ** (attempt - 1)) + random.uniform(0.2, 1.0)
            print(f"⚠️ Attempt {attempt}/{retries} failed: {e}")
            print(f"   Sleeping {backoff:.2f}s then retry...")
            time.sleep(backoff)
    raise RuntimeError(f"❌ Failed after {retries} retries: {url}\nLast error: {last_err}")


# -----------------------------------------
# 5) Parse listing page
# -----------------------------------------
def parse_marketplace_page(html):
    soup = BeautifulSoup(html, "lxml")

    # Find all links to /marketplace/actions/<slug>
    # Some pages have multiple links; we filter by href pattern
    action_links = soup.select('a[href^="/marketplace/actions/"]')

    items = []
    for a in action_links:
        href = (a.get("href") or "").strip()
        if not href.startswith("/marketplace/actions/"):
            continue

        name = a.get_text(strip=True)
        # Skip empty link texts
        if not name or len(name) < 2:
            continue

        full_url = urljoin(BASE, href)

        # Try to extract description from nearby container (<p>)
        desc = ""
        container = a
        for _ in range(7):
            if container is None:
                break
            container = container.parent
            if container and container.name in ("li", "div", "article", "section"):
                p = container.find("p")
                if p:
                    desc = p.get_text(" ", strip=True)
                    break

        items.append({"name": name, "description": desc, "url": full_url})

    # Deduplicate within page by URL
    seen = set()
    unique = []
    for it in items:
        if it["url"] not in seen:
            seen.add(it["url"])
            unique.append(it)

    return unique


# -----------------------------------------
# 6) Scrape function
# -----------------------------------------
def scrape_github_marketplace_actions(target=1000, start_page=1, max_pages=800, delay_range=(1.0, 2.0)):
    session = requests.Session()
    rows = []
    page = start_page

    while len(rows) < target and page <= max_pages:
        url = f"{START_URL}&page={page}"
        html = fetch_with_retries(url, session=session, retries=4, timeout=30)
        parsed = parse_marketplace_page(html)

        if not parsed:
            print(f"🛑 No items parsed on page {page}. Stopping.")
            break

        added = 0
        for it in parsed:
            it["page_number"] = page
            rows.append(it)
            added += 1
            if len(rows) >= target:
                break

        print(f"✅ Page {page}: +{added} items | total={len(rows)}")
        time.sleep(random.uniform(*delay_range))
        page += 1

    return pd.DataFrame(rows).head(target)


# -----------------------------------------
# 7) Run scraping + save raw
# -----------------------------------------
df_raw = scrape_github_marketplace_actions(
    target=TARGET_ROWS,
    start_page=START_PAGE,
    max_pages=MAX_PAGES,
    delay_range=DELAY_RANGE
)

print("\nRaw scrape shape:", df_raw.shape)
display(df_raw.head())

RAW_OUT = "marketplace_actions_1000_raw.csv"
df_raw.to_csv(RAW_OUT, index=False)
print(f"💾 Saved raw data to: {RAW_OUT}")


# -----------------------------------------
# 8) Data Quality (DQ)
# -----------------------------------------
def run_data_quality(df):
    df = df.copy()

    required = ["name", "description", "url", "page_number"]
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    df["name"] = df["name"].fillna("").astype(str).str.strip()
    df["description"] = df["description"].fillna("").astype(str).str.strip()
    df["url"] = df["url"].fillna("").astype(str).str.strip()
    df["page_number"] = pd.to_numeric(df["page_number"], errors="coerce")

    # URL validity
    df["url_valid"] = df["url"].str.match(r"^https://github\.com/marketplace/actions/[^/\s]+/?$", na=False)

    # Dedup by URL
    before = len(df)
    df = df.drop_duplicates(subset=["url"], keep="first").copy()
    after = len(df)

    # Missing checks
    df["name_missing"] = df["name"].eq("") | df["name"].str.lower().eq("nan")
    df["desc_missing"] = df["description"].eq("") | df["description"].str.lower().eq("nan")
    df["page_missing"] = df["page_number"].isna()

    # Fill missing descriptions
    df.loc[df["desc_missing"], "description"] = "No description provided."

    report = {
        "rows_after_dedup": int(len(df)),
        "duplicates_removed": int(before - after),
        "invalid_urls": int((~df["url_valid"]).sum()),
        "missing_names": int(df["name_missing"].sum()),
        "missing_pages": int(df["page_missing"].sum()),
        "missing_descriptions_before_fill": int(df["desc_missing"].sum()),
    }

    return df, report


df_dq, dq_report = run_data_quality(df_raw)

print("\n📋 Data Quality Report")
for k, v in dq_report.items():
    print(f" - {k}: {v}")

display(df_dq.head())


# -----------------------------------------
# 9) Text Preprocessing (tokenize, stopwords, lemmatize)
# -----------------------------------------
def clean_text(s: str) -> str:
    if pd.isna(s):
        return ""

    s = str(s).lower()
    s = re.sub(r"<[^>]+>", " ", s)              # remove HTML
    s = re.sub(r"[^a-z0-9\s]", " ", s)          # remove special chars
    s = re.sub(r"\s+", " ", s).strip()          # remove extra whitespace

    tokens = word_tokenize(s)                   # ✅ works now (punkt_tab fixed)
    tokens = [t for t in tokens if t not in STOP and len(t) > 2]
    tokens = [LEM.lemmatize(t) for t in tokens]

    return " ".join(tokens)

df_dq["text_raw"] = (df_dq["name"].fillna("") + " " + df_dq["description"].fillna("")).str.strip()
df_dq["text_clean"] = df_dq["text_raw"].apply(clean_text)

display(df_dq[["name", "description", "text_clean"]].head())


# -----------------------------------------
# 10) Save cleaned CSV
# -----------------------------------------
CLEAN_OUT = "marketplace_actions_1000_clean.csv"
df_dq.to_csv(CLEAN_OUT, index=False)
print(f"💾 Saved cleaned data to: {CLEAN_OUT}")


# -----------------------------------------
# 11) TF-IDF top keywords (optional)
# -----------------------------------------
try:
    vec = TfidfVectorizer(min_df=3)
    X = vec.fit_transform(df_dq["text_clean"].fillna(""))

    avg = np.asarray(X.mean(axis=0)).ravel()
    terms = np.array(vec.get_feature_names_out())

    top_idx = avg.argsort()[::-1][:10]
    top_terms_df = pd.DataFrame({"term": terms[top_idx], "avg_tfidf": avg[top_idx]}).reset_index(drop=True)

    print("\n🧠 Top TF-IDF terms (average):")
    display(top_terms_df)
except Exception as e:
    print("⚠️ TF-IDF skipped due to:", e)


# -----------------------------------------
# 12) Download outputs (Colab)
# -----------------------------------------
from google.colab import files
files.download(RAW_OUT)
files.download(CLEAN_OUT)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


✅ Setup complete (libraries + NLTK data).
✅ Page 1: +20 items | total=20
✅ Page 2: +20 items | total=40
✅ Page 3: +20 items | total=60
✅ Page 4: +20 items | total=80
✅ Page 5: +20 items | total=100
✅ Page 6: +20 items | total=120
✅ Page 7: +20 items | total=140
✅ Page 8: +20 items | total=160
✅ Page 9: +20 items | total=180
✅ Page 10: +20 items | total=200
✅ Page 11: +20 items | total=220
✅ Page 12: +20 items | total=240
✅ Page 13: +20 items | total=260
✅ Page 14: +20 items | total=280
✅ Page 15: +20 items | total=300
✅ Page 16: +20 items | total=320
✅ Page 17: +20 items | total=340
✅ Page 18: +20 items | total=360
✅ Page 19: +20 items | total=380
✅ Page 20: +20 items | total=400
✅ Page 21: +20 items | total=420
✅ Page 22: +20 items | total=440
✅ Page 23: +20 items | total=460
✅ Page 24: +20 items | total=480
✅ Page 25: +20 items | total=500
✅ Page 26: +20 items | total=520
✅ Page 27: +20 items | total=540
✅ Page 28: +20 items | total=560
✅ Page 29: +20 items | total=580
✅ Page 30: +20

,name,description,url,page_number
0,TruffleHog OSS,Find and verify leaked credentials in your sou...,https://github.com/marketplace/actions/truffle...,1
1,Metrics embed,An infographics generator with 40+ plugins and...,https://github.com/marketplace/actions/metrics...,1
2,yq - portable yaml processor,"create, read, update, delete, merge, validate ...",https://github.com/marketplace/actions/yq-port...,1
3,Super-Linter,Super-linter is a ready-to-run collection of l...,https://github.com/marketplace/actions/super-l...,1
4,Rebuild Armbian and Kernel,"Support Amlogic, Rockchip and Allwinner boxes",https://github.com/marketplace/actions/rebuild...,1


💾 Saved raw data to: marketplace_actions_1000_raw.csv

📋 Data Quality Report
 - rows_after_dedup: 995
 - duplicates_removed: 5
 - invalid_urls: 0
 - missing_names: 0
 - missing_pages: 0
 - missing_descriptions_before_fill: 1


,name,description,url,page_number,url_valid,name_missing,desc_missing,page_missing
0,TruffleHog OSS,Find and verify leaked credentials in your sou...,https://github.com/marketplace/actions/truffle...,1,True,False,False,False
1,Metrics embed,An infographics generator with 40+ plugins and...,https://github.com/marketplace/actions/metrics...,1,True,False,False,False
2,yq - portable yaml processor,"create, read, update, delete, merge, validate ...",https://github.com/marketplace/actions/yq-port...,1,True,False,False,False
3,Super-Linter,Super-linter is a ready-to-run collection of l...,https://github.com/marketplace/actions/super-l...,1,True,False,False,False
4,Rebuild Armbian and Kernel,"Support Amlogic, Rockchip and Allwinner boxes",https://github.com/marketplace/actions/rebuild...,1,True,False,False,False


...

,term,avg_tfidf
0,action,0.079929
1,github,0.068280
2,run,0.038474
3,setup,0.031993
4,code,0.025743
5,file,0.025121
6,request,0.025110
7,deploy,0.024631
8,build,0.024462
9,pull,0.023452


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


In [ ]:
# =========================================
# ONE-SINGLE-CELL GOOGLE COLAB CODE
# Tweepy Hashtag Scraper (ML/AI) + Cleaning + Data Quality + CSV Download
# Handles: 402 Payment Required (no credits)
# =========================================

!pip -q install tweepy pandas

import re, time, random
import pandas as pd
import tweepy
from google.colab import files

# ----------------------------
# 1) PUT YOUR BEARER TOKEN
# ----------------------------
BEARER_TOKEN = "AAAAAAAAAAAAAAAAAAAAABfb7wEAAAAAEunQPEN%2B8v3hbTYQmcfUt9%2FoLR4%3DdrlAl1jGSrp0ew2dbejtRC22u7cAmVGXYFraI1O5AHrNLIvyH8"  # must be your real token

# ----------------------------
# 2) SCRAPE CONFIG
# ----------------------------
HASHTAGS = ["#machinelearning", "#artificialintelligence", "#AI", "#ML"]
MAX_TOTAL_PER_HASHTAG = 100   # keep small to reduce usage; still needs credits if endpoint is paid
SLEEP_BETWEEN_PAGES = 1.0

def build_query(hashtag: str) -> str:
    return f'{hashtag} -is:retweet lang:en'

def scrape_v2_recent(client: tweepy.Client, hashtag: str, max_total: int):
    query = build_query(hashtag)
    rows = []
    next_token = None

    while len(rows) < max_total:
        remaining = max_total - len(rows)
        max_results = 100 if remaining > 100 else remaining

        resp = client.search_recent_tweets(
            query=query,
            max_results=max_results,
            tweet_fields=["id", "text"],
            expansions=["author_id"],
            user_fields=["username"],
            next_token=next_token
        )

        if resp.data is None:
            break

        users = {u["id"]: u["username"] for u in (resp.includes.get("users", []) if resp.includes else [])}

        for t in resp.data:
            rows.append({
                "hashtag": hashtag,
                "tweet_id": str(t.id),
                "username": users.get(str(t.author_id), ""),
                "text": t.text
            })
            if len(rows) >= max_total:
                break

        meta = resp.meta or {}
        next_token = meta.get("next_token")
        if not next_token:
            break

        time.sleep(SLEEP_BETWEEN_PAGES)

    return rows

# ----------------------------
# 3) PART 2 — CLEANING + DQ
# ----------------------------
REQUIRED_COLS = ["tweet_id", "username", "text", "hashtag"]

def clean_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s)
    s = re.sub(r"http\S+|www\.\S+", " ", s)  # URLs
    s = re.sub(r"@\w+", " ", s)             # mentions
    s = s.replace("#", "")                  # hashtag symbol only
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s

def dq_and_clean(df: pd.DataFrame):
    df = df.copy()

    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    for c in REQUIRED_COLS:
        df[c] = df[c].fillna("").astype(str).str.strip()

    df["clean_text"] = df["text"].apply(clean_text)

    # DQ flags
    df["tweet_id_missing"] = df["tweet_id"].eq("")
    df["username_missing"] = df["username"].eq("")
    df["text_missing"] = df["clean_text"].eq("")

    before = len(df)
    df = df.drop_duplicates(subset=["tweet_id"], keep="first").copy()
    after_dedup = len(df)

    df = df[~(df["tweet_id_missing"] | df["username_missing"] | df["text_missing"])].copy()
    after_final = len(df)

    report = {
        "rows_start": int(before),
        "rows_after_dedup": int(after_dedup),
        "rows_after_final_clean": int(after_final),
        "duplicates_removed": int(before - after_dedup),
        "rows_removed_missing_required": int(after_dedup - after_final),
    }

    df = df.drop(columns=["tweet_id_missing", "username_missing", "text_missing"], errors="ignore")
    return df, report

# ----------------------------
# 4) TRY SCRAPE (handles 402)
# ----------------------------
df_raw = pd.DataFrame()

try:
    if not BEARER_TOKEN or "PASTE_" in BEARER_TOKEN:
        raise ValueError("Please paste your real BEARER_TOKEN first.")

    client = tweepy.Client(bearer_token=BEARER_TOKEN, wait_on_rate_limit=True)
    print("✅ Tweepy v2 Client created.")

    all_rows = []
    for h in HASHTAGS:
        print(f"Scraping {h} ...")
        rows = scrape_v2_recent(client, h, MAX_TOTAL_PER_HASHTAG)
        print(f"  ✅ Collected {len(rows)} tweets for {h}")
        all_rows.extend(rows)

    df_raw = pd.DataFrame(all_rows)
    print("\nRaw dataset shape:", df_raw.shape)
    display(df_raw.head())

except tweepy.HTTPException as e:
    # 402 payment required / no credits
    msg = str(e)
    if "402" in msg or "Payment Required" in msg or "does not have any credits" in msg:
        print("\n❌ X API returned: 402 Payment Required (no credits).")
        print("You cannot use search_recent_tweets on this account until you add credits / upgrade your API access.")
        print("✅ You can still complete PART 2 by uploading an existing tweets CSV (tweet_id, username, text, hashtag).")
    else:
        print("\n❌ Tweepy HTTPException:", e)

except Exception as e:
    print("\n❌ Error:", e)

# ----------------------------
# 5) FALLBACK: Upload CSV if scraping failed
# ----------------------------
if df_raw.empty:
    print("\n📌 Upload a CSV file now (must include columns: tweet_id, username, text, hashtag).")
    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError("No file uploaded. Please upload a CSV to proceed with PART 2.")

    # Load the first uploaded file
    fname = list(uploaded.keys())[0]
    df_raw = pd.read_csv(fname)
    print(f"✅ Loaded uploaded file: {fname} | shape={df_raw.shape}")
    display(df_raw.head())

# ----------------------------
# 6) RUN PART 2 + SAVE + DOWNLOAD
# ----------------------------
df_clean, dq_report = dq_and_clean(df_raw)

print("\n📋 Data Quality Report:")
for k, v in dq_report.items():
    print(f" - {k}: {v}")

OUTFILE = "tweets_ml_ai_clean.csv"
df_clean.to_csv(OUTFILE, index=False)
print(f"\n✅ Saved CSV: {OUTFILE} | rows={len(df_clean)}")

files.download(OUTFILE)

✅ Tweepy v2 Client created.
Scraping #machinelearning ...

❌ X API returned: 402 Payment Required (no credits).
You cannot use search_recent_tweets on this account until you add credits / upgrade your API access.
✅ You can still complete PART 2 by uploading an existing tweets CSV (tweet_id, username, text, hashtag).

📌 Upload a CSV file now (must include columns: tweet_id, username, text, hashtag).


# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

I took almost 4 days to complete the assignment. it was quiet challenging and as well time consuming. there are several challenges when i have tried coding the the Question 1. i have learned different aspects from this assignment like efficient scraping and data preprocessing. it was pretty tough as i am new to python but ill try to catchup. And i felt i personally needed more time to give efficient results.